In [2]:
import os

import io
import json
import tarfile

import numpy as np

from numcodecs import JSON
import zarr

from skimage.segmentation import relabel_sequential

from tqdm import tqdm
import glob

def load_trks(filename):
    """Load a trk/trks file.

    Args:
        filename (str or BytesIO): full path to the file including .trk/.trks
            or BytesIO object with trk file data

    Returns:
        dict: A dictionary with raw, tracked, and lineage data.
    """

    if isinstance(filename, io.BytesIO):
        kwargs = {'fileobj': filename}
    else:
        kwargs = {'name': filename}

    with tarfile.open(mode='r', **kwargs) as trks:

        # numpy can't read these from disk...
        with io.BytesIO() as array_file:
            array_file.write(trks.extractfile('raw.npy').read())
            array_file.seek(0)
            raw = np.load(array_file)

        with io.BytesIO() as array_file:
            array_file.write(trks.extractfile('tracked.npy').read())
            array_file.seek(0)
            tracked = np.load(array_file)

        # trks.extractfile opens a file in bytes mode, json can't use bytes.
        try:
            trk_data = trks.getmember('lineages.json')
        except KeyError:
            try:
                trk_data = trks.getmember('lineage.json')
            except KeyError:
                raise ValueError('Invalid .trk file, no lineage data found.')

        lineages = json.loads(trks.extractfile(trk_data).read().decode())
        lineages = lineages if isinstance(lineages, list) else [lineages]

        # JSON only allows strings as keys, so convert them back to ints
        for i, tracks in enumerate(lineages):
            lineages[i] = {int(k): v for k, v in tracks.items()}

    return {'lineages': lineages, 'X': raw, 'y': tracked}

def convert_trk_to_zarr(filename, out_dir=None):

    if out_dir is None:
        file_dir = os.path.dirname(filename)
        split = os.path.splitext(os.path.basename(filename))[0]
        output_file = os.path.join(file_dir, split) + '.zarr'

    data = load_trks(filename)

    X = data['X']
    y = data['y']
    lineages = data['lineages']


    z = zarr.open(output_file)

    lin_out = np.empty(1, dtype=object)
    lin_out[0] = lineages

    zlin = z.array('lineages', lin_out, shape=lin_out.shape, dtype=object, object_codec=JSON())
    zX = z.array('X', X, shape=X.shape, chunks = (1, 1, X.shape[2], X.shape[3], 1))
    zy = z.array('y', y, shape=y.shape, chunks = (1, 1, y.shape[2], y.shape[3], 1))

    # return z


for filename in glob.glob('data/DynamicNuclearNet-tracking-v1_0/*.trks'):
    print(f"Converting {os.path.basename(filename)}")
    convert_trk_to_zarr(filename)

Converting test.trks
Converting val.trks
Converting train.trks


In [ ]:
def is_valid_lineage(y, lineage):
    """Check if a cell lineage of a single movie is valid.

    Daughter cells must exist in the frame after the parent's final frame.

    Args:
        y (numpy.array): The 3D label mask.
        lineage (dict): The cell lineages for a single movie.

    Returns:
        bool: Whether or not the lineage is valid.
    """
    all_cells = np.unique(y)
    all_cells = set([c for c in all_cells if c])

    is_valid = True

    # every lineage should have valid fields
    for cell_label, cell_lineage in lineage.items():
        cell_label = int(cell_label)
        # Get last frame of parent
        if cell_label not in all_cells:
            warnings.warn('Cell {} not found in the label image.'.format(
                cell_label))
            is_valid = False
            continue

        # any cells leftover are missing lineage
        all_cells.remove(cell_label)

        # validate `frames`
        y_true = np.any(y == cell_label, axis=(1, 2))
        y_index = y_true.nonzero()[0]
        frames = list(y_index)
        if frames != cell_lineage['frames']:
            warnings.warn('Cell {} has invalid frames'.format(cell_label))
            is_valid = False
            continue  # no need to test further

        last_parent_frame = cell_lineage['frames'][-1]

        for daughter in cell_lineage['daughters']:
            daughter=str(daughter)
            if daughter not in lineage:
                warnings.warn('Lineage {} has daughter {} not in lineage'.format(
                    cell_label, daughter))
                is_valid = False
                continue  # no need to test further

            # get first frame of daughter
            try:
                first_daughter_frame = lineage[daughter]['frames'][0]
            except IndexError:  # frames is empty?
                warnings.warn('Daughter {} has no frames'.format(daughter))
                is_valid = False
                continue  # no need to test further

            # Check that daughter's start frame is one larger than parent end frame
            if first_daughter_frame - last_parent_frame != 1:
                warnings.warn('Lineage {} has daughter {} in a '
                              'non-subsequent frame.'.format(
                                  cell_label, daughter))
                is_valid = False
                continue  # no need to test further

        # TODO: test parent in lineage
        parent = cell_lineage.get('parent')
        if parent:
            parent = str(parent)
            try:
                parent_lineage = lineage[parent]
            except KeyError:
                warnings.warn('Parent {} is not present in the lineage'.format(
                    cell_lineage['parent']))
                is_valid = False
                continue  # no need to test further
            try:
                last_parent_frame = parent_lineage['frames'][-1]
                first_daughter_frame = cell_lineage['frames'][0]
            except IndexError:  # frames is empty?
                warnings.warn('Cell {} has no frames'.format(parent))
                is_valid = False
                continue  # no need to test further
            # Check that daughter's start frame is one larger than parent end frame
            if first_daughter_frame - last_parent_frame != 1:
                warnings.warn(
                    'Cell {} ends in frame {} but daughter {} first '
                    'appears in frame {}.'.format(
                        parent, last_parent_frame, cell_label,
                        first_daughter_frame))
                is_valid = False
                continue  # no need to test further

    if all_cells:  # all cells with lineages should be removed
        warnings.warn('Cells missing their lineage: {}'.format(
            list(all_cells)))
        is_valid = False

    return is_valid  # if unchanged, all cell lineages are valid!


In [ ]:
def relabel_sequential_lineage(y, lineage):
    """Ensure the lineage information is sequentially labeled.

    Args:
        y (np.array): Annotated z-stack of image labels.
        lineage (dict): Lineage data for y.

    Returns:
        tuple(np.array, dict): The relabeled array and corrected lineage.
    """
    
    y_relabel, fw, _ = relabel_sequential(y)

    new_lineage = {}

    cell_ids = np.unique(y)
    cell_ids = cell_ids[cell_ids != 0]
    cell_ids = cell_ids.tolist()
    
    for cell_id in cell_ids:
        
        new_cell_id = int(fw[cell_id])

        new_lineage[new_cell_id] = {}

        # Fix label
        # TODO: label == track ID?
        new_lineage[new_cell_id]['label'] = new_cell_id
        cell_id = str(cell_id)
        # Fix parent
        parent = lineage[cell_id]['parent']
        new_parent = int(fw[parent]) if parent is not None else parent
        new_lineage[new_cell_id]['parent'] = new_parent

        # Fix daughters
        daughters = lineage[cell_id]['daughters']
        new_lineage[new_cell_id]['daughters'] = []
        for d in daughters:
            new_daughter = int(fw[d])
            if not new_daughter:  # missing labels get mapped to 0
                warnings.warn('Cell {} has daughter {} which is not found '
                              'in the label image `y`.'.format(cell_id, d))
            else:
                new_lineage[new_cell_id]['daughters'].append(new_daughter)
                
        # Fix frames
        y_true = np.any(y == np.int64(cell_id), axis=(1, 2))
        y_index = np.nonzero(y_true)[0]

        new_lineage[new_cell_id]['frames'] = y_index.tolist()

    return y_relabel, new_lineage

In [ ]:
y = z['y'][:]
lineages = z['lineages'][0]
print(len(lineages))
for i in range(len(lineages)):

    is_valid_lineage(y[i], lineages[i])
    y_relabel, new_lineage = relabel_sequential_lineage(y[i], lineages[i])


12
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30

KeyboardInterrupt: 

In [8]:
import os

import io
import json
import tarfile

import numpy as np

from numcodecs import JSON
import zarr

from skimage.segmentation import relabel_sequential

from tqdm import tqdm
import glob

def get_max_frames(lineage):

    movie_len = 0

    for k, v in lineage.items():
        curr_max = np.max(v['frames'])
        if curr_max >= movie_len:
            movie_len = curr_max

    return movie_len

import matplotlib.pyplot as plt
X = zarr.open('data/DynamicNuclearNet-tracking-v1_0/train.zarr')['X']
y = zarr.open('data/DynamicNuclearNet-tracking-v1_0/train.zarr')['y']

lineages = zarr.open('data/DynamicNuclearNet-tracking-v1_0/train.zarr')['lineages'][0]
print(X.shape)

mit_sum = 0
same_sum = 0
no_sum = 0

for batch in lineages:
    
    max_label = len(batch)
    max_frames = X.shape[1]
    print(max_label, max_frames)
    labels = np.zeros((max_frames, max_label, max_label))

    for label, track in batch.items():

        # The frames where each track is linked to itself (encoded as 1)
        frames_linked = track['frames'][:-1]

        #Identity label
        identity_label = int(track['label'])-1

        # Place all identities
        labels[frames_linked, identity_label, identity_label] = 1

        # Place all mitotic frames as linkages
        if track['frame_div']:

            frame_divided = track['frame_div']-1
            daughter_label = track['daughters']
            

            daughters_tuple = [daughter-1 for daughter in daughter_label]
            # print(daughters_tuple)
            labels[frame_divided, identity_label, daughters_tuple] = 2
    
    mit_sum += np.sum(labels == 2)/labels.size
    same_sum += np.sum(labels == 1)/labels.size
    no_sum += np.sum(labels == 0)/labels.size

print(1/(mit_sum/len(lineages)), 1/(same_sum/len(lineages)), 1/(no_sum/len(lineages)))

(91, 71, 584, 600, 1)
46 71
36 71
21 71
19 71
33 71
61 71
47 71
43 71
48 71
40 71
45 71
45 71
24 71
69 71
33 71
34 71
36 71
26 71
27 71
42 71
52 71
31 71
32 71
40 71
32 71
24 71
17 71
37 71
48 71
56 71
48 71
50 71
43 71
41 71
23 71
26 71
63 71
33 71
34 71
26 71
28 71
22 71
25 71
12 71
5 71
132 71
76 71
168 71
107 71
102 71
134 71
100 71
94 71
150 71
350 71
333 71
290 71
375 71
366 71
405 71
200 71
111 71
185 71
202 71
194 71
169 71
340 71
183 71
255 71
143 71
101 71
206 71
260 71
133 71
184 71
153 71
232 71
255 71
253 71
148 71
236 71
247 71
323 71
179 71
244 71
145 71
164 71
130 71
251 71
251 71
157 71
26770.358769655955 98.02656496939109 1.0103445859110196
